### In this notebook we show how the model to model evaluation module works

In [ ]:
# STENCIL DEFINITIONS
# ==============================================================================

# Updated based on your complete stencil list
START_EVENT_STENCILS = [
    "StartNoneEvent",
    "StartMessageEvent",
    "StartTimerEvent",
    "StartSignalEvent",
    "StartConditionalEvent",
    "StartErrorEvent",
    "StartEscalationEvent",
    "StartCompensationEvent",
    "StartMultipleEvent",
    "StartParallelMultipleEvent",
]

END_EVENT_STENCILS = [
    "EndNoneEvent",
    "EndMessageEvent",
    "EndTerminateEvent",
    "EndErrorEvent",
    "EndEscalationEvent",
    "EndCompensationEvent",
    "EndCancelEvent",
    "EndMultipleEvent",
    "EndSignalEvent",
]

INTERMEDIATE_EVENT_STENCILS = [
    "IntermediateEvent",
    "IntermediateMessageEventCatching",
    "IntermediateMessageEventThrowing",
    "IntermediateTimerEvent",
    "IntermediateErrorEvent",
    "IntermediateConditionalEvent",
    "IntermediateEscalationEvent",
    "IntermediateEscalationEventThrowing",
    "IntermediateSignalEventThrowing",
    "IntermediateSignalEventCatching",
    "IntermediateCompensationEventCatching",
    "IntermediateCompensationEventThrowing",
    "IntermediateCancelEvent",
    "IntermediateMultipleEventCatching",
    "IntermediateMultipleEventThrowing",
    "IntermediateParallelMultipleEventCatching",
    "IntermediateLinkEventThrowing",
    "IntermediateLinkEventCatching",
]

GATEWAY_STENCILS = [
    "Exclusive_Databased_Gateway",
    "ParallelGateway",
    "InclusiveGateway",
    "ComplexGateway",
    "EventbasedGateway",
]

ACTIVITY_STENCILS = ["Task", "Subprocess", "CollapsedSubprocess", "EventSubprocess", "CollapsedEventSubprocess"]

# All flow nodes (elements that can have sequence flows)
FLOW_NODE_STENCILS = (
    START_EVENT_STENCILS + END_EVENT_STENCILS + INTERMEDIATE_EVENT_STENCILS + GATEWAY_STENCILS + ACTIVITY_STENCILS
)

# Supporting elements
SUPPORTING_STENCILS = [
    "DataObject",
    "DataStore",
    "TextAnnotation",
    "Group",
    "Message",
    "Association_Unidirectional",
    "Association_Undirected",
    "Association_Bidirectional",
    "MessageFlow",
    "ITSystem",
]

from collections import Counter


def all_stencils(bpmn_models):
    all_stencils = []
    for i in final_count(bpmn_models):
        if i[0] not in all_stencils:
            all_stencils.append(i[0])
    return all_stencils


def count_stencil_ids(obj):
    """Recursively counts occurrences of all stencil IDs in a Signavio model tree."""
    counts = Counter()

    if isinstance(obj, dict):
        # If this object has a 'stencil' with an 'id', count it
        stencil = obj.get("stencil")
        if isinstance(stencil, dict) and "id" in stencil:
            counts[stencil["id"]] += 1
        # Recurse into any childShapes
        childshapes = obj.get("childShapes")
        if isinstance(childshapes, list):
            for child in childshapes:
                counts.update(count_stencil_ids(child))
        # Optionally, you can search other dict values in case the model is nonstandard
        # for v in obj.values():
        #    counts.update(count_stencil_ids(v))

    elif isinstance(obj, list):
        for item in obj:
            counts.update(count_stencil_ids(item))

    return counts


def final_count(bpmn_models):
    """
    Counts all stencil IDs across multiple BPMN models and returns a sorted list of counts.
    """

    final_dict = {}
    for i in bpmn_models:
        result = dict(count_stencil_ids(i))
        for m in result.keys():
            if m in final_dict:
                final_dict[m] += 1
            else:
                final_dict[m] = 1
    return sorted(final_dict.items(), key=lambda x: x[1], reverse=True)


def extract_elements_by_stencil_ids(model, stencil_ids):
    """
    Recursively extract all elements matching any of the given stencil IDs.

    Args:
        model: The BPMN model (JSON structure)
        stencil_ids: List of stencil IDs to match

    Returns:
        List of matching elements
    """
    results = []
    shapes = model.get("childShapes", [])
    for shape in shapes:
        if shape.get("stencil", {}).get("id") in stencil_ids:
            results.append(shape)
        if "childShapes" in shape:
            results.extend(extract_elements_by_stencil_ids(shape, stencil_ids))
    return results

In [ ]:
import sys, json

sys.path.append("../")
sys.path.append("../model_evaluation/")

In [ ]:
# load some examples from examples folder in Signavio json format
filename_ground_truth = f"../examples/E_j04.json"
with open(filename_ground_truth, "r") as infile:
    E4 = json.load(infile)


filename_generated = f"../examples/E_j04_4.bpmn2 _ Signavio.json"
with open(filename_generated, "r") as infile:
    E4_1 = json.load(infile)


filename_generated = f"../examples/process_complex.json"
with open(filename_generated, "r") as infile:
    pc = json.load(infile)

filename_generated = f"../examples/misc_booking_flight_tickets.json"
with open(filename_generated, "r") as infile:
    misc_loan_ft = json.load(infile)

filename_generated = f"../examples/misc_credit_quote_creation.json"
with open(filename_generated, "r") as infile:
    misc_loan_credit = json.load(infile)

filename_generated = f"../examples/Adrians_ex.json"
with open(filename_generated, "r") as infile:
    adrians_ex = json.load(infile)

filename_generated = f"../examples/simplemodel.json"
with open(filename_generated, "r") as infile:
    simple_model = json.load(infile)

In [ ]:
count_stencil_ids(misc_loan_ft)

In [ ]:
from BPMN_conversion import BPMNConverter

# from bpmn_schema_helper import BPMNConverter as original_BPMNConverter

In [ ]:
import json

# E4_json = BPMNConverter.convert(E4).to_json()

# E4_1_json = BPMNConverter.convert(E4_1).to_json()

misc_bft_json = BPMNConverter.convert(misc_loan_ft).to_json()
# misc_bft_json_original = original_BPMNConverter.convert(E4_1).to_json()

misc_credit_json = BPMNConverter.convert(misc_loan_credit).to_json()

pc_json = BPMNConverter.convert(pc).to_json()

# simple = BPMNConverter.convert(simple_model).to_json()
# pc_original_json = original_BPMNConverter.convert(pc).to_json()

# adrians_json = BPMNConverter.convert(adrians_ex).to_json()
# write E4_json to file
# with open("../E4_minimal.json", "w") as outfile:
#     E4_1_json = BPMNConverter.convert(misc_ft)
#     json.dump(json.loads(E4_1_json.to_json()), outfile, indent=4)

In [ ]:
json.loads(misc_bft_json)
from bpmn_sets import extract_bpmn_sets

extract_bpmn_sets(json.loads(misc_bft_json))

## Complete BPMN Model Comparison Workflow

This section demonstrates the complete workflow for comparing two BPMN models using the evaluation framework.

In [ ]:
# ==============================================================================
# COMPLETE BPMN COMPARISON PIPELINE - CORRECT ORDER
# ==============================================================================
#
# Pipeline Steps:
# 1. Load Signavio JSON files
# 2. Convert to minimal BPMN format (BPMN_conversion)
# 3. Normalize atomic names (optional but recommended - bpmn_normalization)
# 4. Calculate similarity (bpmn_similarity)
#
# Note: Steps 2 and 3 happen BEFORE similarity calculation!
# ==============================================================================

import sys
import json

sys.path.append("../model_evaluation/")

from BPMN_conversion import BPMNConverter
from bpmn_normalization import normalize_atomic_names
from string_similarity import bert_cosine_optimized
from bpmn_similarity import calculate_bpmn_similarity

# STEP 1: Load Signavio JSON files
print("Step 1: Loading Signavio JSON files...")
with open("../examples/misc_booking_flight_tickets.json", "r") as f:
    ground_truth_signavio = json.load(f)

with open("../examples/test.json", "r") as f:
    generated_signavio = json.load(f)

# STEP 2: Convert to minimal BPMN format
print("\nStep 2: Converting Signavio JSON to minimal BPMN format...")
ground_truth_minimal = BPMNConverter.convert(ground_truth_signavio)
generated_minimal = BPMNConverter.convert(generated_signavio)

# Parse to dict for normalization and comparison
ground_truth_dict = json.loads(ground_truth_minimal.to_json())
generated_dict = json.loads(generated_minimal.to_json())

print(
    f"  Ground Truth: {len(ground_truth_dict['activities'])} activities, "
    f"{len(ground_truth_dict['events'])} events, "
    f"{len(ground_truth_dict['gateways'])} gateways"
)
print(
    f"  Generated:    {len(generated_dict['activities'])} activities, "
    f"{len(generated_dict['events'])} events, "
    f"{len(generated_dict['gateways'])} gateways"
)

# STEP 3: Normalize atomic names (IMPORTANT: before similarity calculation!)
print("\nStep 3: Normalizing atomic names using semantic similarity...")
print("  (This aligns element names like 'Book flight' ↔ 'Book a flight')")
generated_normalized, mappings = normalize_atomic_names(
    ground_truth_dict, generated_dict, bert_cosine_optimized, threshold=0.8
)

# if mappings:
#     print(f"  Applied {sum(len(v) for v in mappings.values())} name mappings:")
#     for atomic_type, mapping in mappings.items():
#         if mapping:
#             print(f"    {atomic_type}: {len(mapping)} mappings")
#             # Show first 2 examples
#             for i, (old_name, new_name) in enumerate(list(mapping.items())[:2]):
#                 print(f"      '{old_name}' → '{new_name}'")
# else:
#     print("  No semantic mappings needed (names already match)")

# STEP 4: Calculate similarity (AFTER normalization!)
print("\nStep 4: Calculating BPMN similarity...")
similarity_results = calculate_bpmn_similarity(
    ground_truth_dict, generated_normalized, method="dice"  # Use normalized version!
)

print(f"\n{'='*60}")
print(f"OVERALL SIMILARITY: {similarity_results['overall']:.3f} ({similarity_results['overall']*100:.1f}%)")
print(f"{'='*60}")
print("\nHigh-Level Scores:")
for category, score in similarity_results["high_level_scores"].items():
    weight = similarity_results["weights_used"][category]
    print(f"  {category:20s} (weight={weight:4.0%}): {score:.3f} ({score*100:5.1f}%)")

### Comparison: With vs Without Normalization

This demonstrates the impact of semantic normalization on similarity scores.

In [ ]:
# Compare similarity WITH normalization vs WITHOUT normalization
print("Comparing: WITH normalization vs WITHOUT normalization\n")

# WITHOUT normalization (direct comparison)
similarity_without_norm = calculate_bpmn_similarity(
    ground_truth_dict, generated_dict, method="dice"  # Original, not normalized
)

# WITH normalization (what we calculated above)
similarity_with_norm = similarity_results

print(f"{'Category':<20} {'Without Norm':>12} {'With Norm':>12} {'Improvement':>12}")
print("-" * 60)

for category in ["structural", "flows", "organizational", "subprocess"]:
    without = similarity_without_norm["high_level_scores"][category]
    with_norm = similarity_with_norm["high_level_scores"][category]
    improvement = with_norm - without
    print(f"{category:<20} {without:>12.3f} {with_norm:>12.3f} {improvement:>+12.3f}")

print("-" * 60)
overall_without = similarity_without_norm["overall"]
overall_with = similarity_with_norm["overall"]
overall_improvement = overall_with - overall_without

print(f"{'OVERALL':<20} {overall_without:>12.3f} {overall_with:>12.3f} {overall_improvement:>+12.3f}")
print(
    f"\nConclusion: Normalization {'improves' if overall_improvement > 0 else 'does not improve'} similarity by "
    f"{abs(overall_improvement)*100:.2f} percentage points"
)

In [ ]:
# ==============================================================================
# ALTERNATIVE: Quick comparison without normalization
# ==============================================================================
# Use this when you want a fast comparison and know names are already aligned,
# or when you're comparing against a reference without semantic variations.

# Load different models for this example
with open("../examples/E_j04.json", "r") as f:
    model1_signavio = json.load(f)

with open("../examples/E_j04_4.bpmn2 _ Signavio.json", "r") as f:
    model2_signavio = json.load(f)

# Convert to minimal BPMN
model1_dict = json.loads(BPMNConverter.convert(model1_signavio).to_json())
model2_dict = json.loads(BPMNConverter.convert(model2_signavio).to_json())

print("Model 1:")
print(f"  Activities: {len(model1_dict['activities'])}")
print(f"  Events: {len(model1_dict['events'])}")
print(f"  Gateways: {len(model1_dict['gateways'])}")

print("\nModel 2:")
print(f"  Activities: {len(model2_dict['activities'])}")
print(f"  Events: {len(model2_dict['events'])}")
print(f"  Gateways: {len(model2_dict['gateways'])}")

# Direct similarity calculation (no normalization)
quick_similarity = calculate_bpmn_similarity(model1_dict, model2_dict, method="dice")

print(f"\nOverall Similarity: {quick_similarity['overall']:.3f}")
print("\nNote: This skips normalization. Use the full pipeline above for better results.")

In [ ]:
# ==============================================================================
# Using Different Similarity Metrics
# ==============================================================================
# The framework supports: "dice", "jaccard", "precision", "recall", "f1"

# Using the normalized models from above
print("Comparing different similarity metrics:\n")

methods = ["dice", "jaccard", "precision", "recall", "f1"]
results = {}

for method in methods:
    result = calculate_bpmn_similarity(ground_truth_dict, generated_normalized, method=method)
    results[method] = result
    print(f"{method.upper():<12} Overall: {result['overall']:.3f}")

print("\n" + "=" * 60)
print("Metric Descriptions:")
print("  - Dice:      Balanced similarity (2*|A∩B| / |A|+|B|)")
print("  - Jaccard:   Set overlap (|A∩B| / |A∪B|)")
print("  - Precision: How much of generated is correct (|A∩B| / |B|)")
print("  - Recall:    How much of ground truth is captured (|A∩B| / |A|)")
print("  - F1:        Harmonic mean of precision and recall")

In [ ]:
# ==============================================================================
# Using Custom Weights
# ==============================================================================
# Adjust importance of different BPMN aspects based on your evaluation needs

# Example 1: Flow-heavy weighting (control flow matters most)
flow_heavy_weights = {
    "structural": 0.20,  # 20% - element types and names
    "flows": 0.60,  # 60% - sequence and message flows
    "organizational": 0.15,  # 15% - pools and lanes
    "subprocess": 0.05,  # 5%  - subprocess structure
}

# Example 2: Structure-heavy weighting (elements matter most)
structure_heavy_weights = {"structural": 0.60, "flows": 0.25, "organizational": 0.10, "subprocess": 0.05}

# Compare different weightings
print("Comparing different weight configurations:\n")

configs = [("Default", None), ("Flow-Heavy", flow_heavy_weights), ("Structure-Heavy", structure_heavy_weights)]

for name, weights in configs:
    result = calculate_bpmn_similarity(ground_truth_dict, generated_normalized, method="dice", weights=weights)
    print(f"{name:<20} Overall: {result['overall']:.3f}")
    if weights:
        print(
            f"  Weights used: structural={weights['structural']:.0%}, "
            f"flows={weights['flows']:.0%}, "
            f"org={weights['organizational']:.0%}, "
            f"subprocess={weights['subprocess']:.0%}"
        )
    print()

In [ ]:
# ==============================================================================
# SUMMARY: Best Practices for BPMN Comparison
# ==============================================================================

print(
    """
╔═══════════════════════════════════════════════════════════════════════════╗
║                   RECOMMENDED BPMN COMPARISON PIPELINE                    ║
╠═══════════════════════════════════════════════════════════════════════════╣
║                                                                           ║
║  1. Load Signavio JSON                                                    ║
║     └─ json.load() from file                                             ║
║                                                                           ║
║  2. Convert to Minimal BPMN (BPMN_conversion.py)                         ║
║     └─ BPMNConverter.convert(signavio_json)                              ║
║     └─ Normalizes structure, handles subprocesses                        ║
║                                                                           ║
║  3. Normalize Atomic Names (bpmn_normalization.py) *** IMPORTANT ***     ║
║     └─ normalize_atomic_names(model1, model2, bert_cosine_optimized)    ║
║     └─ Aligns semantic variations in element names                       ║
║     └─ Example: "Book flight" ↔ "Book a flight"                         ║
║                                                                           ║
║  4. Extract BPMN Sets (bpmn_sets.py) - happens automatically             ║
║     └─ extract_bpmn_sets() called internally                             ║
║     └─ Separates top-level from subprocess elements                      ║
║                                                                           ║
║  5. Calculate Similarity (bpmn_similarity.py)                            ║
║     └─ calculate_bpmn_similarity(model1, model2_normalized, method)     ║
║     └─ Returns fine/grouped/high-level scores + overall                  ║
║                                                                           ║
╠═══════════════════════════════════════════════════════════════════════════╣
║  WHY NORMALIZATION MATTERS:                                               ║
║  • LLMs generate semantically correct but syntactically varied names     ║
║  • Without normalization: "Book flight" ≠ "Book a flight" (0% match)    ║
║  • With normalization: "Book flight" ≈ "Book a flight" (95%+ match)     ║
║  • Can improve overall similarity by 10-20 percentage points!            ║
╚═══════════════════════════════════════════════════════════════════════════╝
"""
)

## Interactive Weight Configuration & Visualization

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np

print("Calculating similarity scores (this may take a moment)...")
base_result = calculate_bpmn_similarity(ground_truth_dict, generated_normalized, method="dice")
print("✓ Ready!")

has_subprocess = base_result.get("has_expanded_subprocess", False)

output = widgets.Output()

structural_slider = widgets.FloatSlider(
    value=0.35, min=0, max=1, step=0.05, description="Structural:", continuous_update=False
)
flows_slider = widgets.FloatSlider(value=0.45, min=0, max=1, step=0.05, description="Flows:", continuous_update=False)
organizational_slider = widgets.FloatSlider(
    value=0.15, min=0, max=1, step=0.05, description="Organizational:", continuous_update=False
)
subprocess_slider = widgets.FloatSlider(
    value=0.05 if has_subprocess else 0.0,
    min=0,
    max=1 if has_subprocess else 0,
    step=0.05,
    description="Subprocess:",
    continuous_update=False,
    disabled=not has_subprocess,
)


def normalize_weights(structural, flows, organizational, subprocess):
    if not has_subprocess:
        subprocess = 0.0
    total = structural + flows + organizational + subprocess
    if total == 0:
        return 0.35, 0.45, 0.15, 0.05
    return structural / total, flows / total, organizational / total, subprocess / total


def update_visualization(structural, flows, organizational, subprocess):
    structural, flows, organizational, subprocess = normalize_weights(structural, flows, organizational, subprocess)

    weights = {"structural": structural, "flows": flows, "organizational": organizational, "subprocess": subprocess}

    result = base_result.copy()

    overall = (
        result["high_level_scores"]["structural"] * structural
        + result["high_level_scores"]["flows"] * flows
        + result["high_level_scores"]["organizational"] * organizational
        + result["high_level_scores"]["subprocess"] * subprocess
    )

    with output:
        clear_output(wait=True)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        categories = ["Structural", "Flows", "Organizational", "Subprocess"]
        scores = [
            result["high_level_scores"]["structural"],
            result["high_level_scores"]["flows"],
            result["high_level_scores"]["organizational"],
            result["high_level_scores"]["subprocess"],
        ]
        weights_vals = [structural, flows, organizational, subprocess]
        weighted_scores = [s * w for s, w in zip(scores, weights_vals)]

        colors = ["#3498db", "#2ecc71", "#f39c12", "#9b59b6"]
        y_pos = np.arange(len(categories))

        bars = ax1.barh(y_pos, scores, color=colors, alpha=0.6, label="Raw Score")
        bars_w = ax1.barh(y_pos, weighted_scores, color=colors, alpha=1.0, label="Weighted")

        ax1.set_yticks(y_pos)
        ax1.set_yticklabels(categories)
        ax1.set_xlabel("Score")
        ax1.set_xlim(0, 1)
        ax1.set_title(f"Overall Similarity: {overall:.1%}", fontweight="bold", fontsize=13)
        ax1.legend(loc="lower right")
        ax1.grid(axis="x", alpha=0.3)

        for i, (score, weighted) in enumerate(zip(scores, weighted_scores)):
            ax1.text(score + 0.02, i, f"{score:.2f}", va="center", fontsize=9, color="gray")
            if weighted > 0.01:
                ax1.text(weighted + 0.02, i - 0.15, f"{weighted:.2f}", va="center", fontsize=9, fontweight="bold")

        # Right: Fine-grained element breakdown
        elements = [
            "Activities",
            "Events",
            "Gateways",
            "Seq Flows",
            "Msg Flows",
            "Lanes",
            "Subprocess\nNames",
            "Subprocess\nElements",
            "Subprocess\nFlows",
        ]
        element_scores = [
            result["activity_names"],
            result["event_names"],
            result["gateway_names"],
            result["seq_flows_str"],
            result["mes_flows_str"],
            result["lane_names"],
            result["subprocess_names"],
            result["subprocess_elemrefs"],
            result["subprocess_flows"],
        ]
        element_colors = ["#3498db"] * 3 + ["#2ecc71"] * 2 + ["#f39c12"] + ["#9b59b6"] * 3

        y_pos2 = np.arange(len(elements))
        bars2 = ax2.barh(y_pos2, element_scores, color=element_colors, alpha=0.8)

        ax2.set_yticks(y_pos2)
        ax2.set_yticklabels(elements, fontsize=9)
        ax2.set_xlabel("Score")
        ax2.set_xlim(0, 1)
        ax2.set_title("Element-Level Breakdown", fontweight="bold", fontsize=13)
        ax2.grid(axis="x", alpha=0.3)

        for i, score in enumerate(element_scores):
            ax2.text(score + 0.02, i, f"{score:.2f}", va="center", fontsize=8)

        plt.tight_layout()
        plt.show()


interactive_plot = widgets.interactive(
    update_visualization,
    structural=structural_slider,
    flows=flows_slider,
    organizational=organizational_slider,
    subprocess=subprocess_slider,
)

display(interactive_plot, output)